# 01 — Rule-based: trend + breakout + volume + Monte Carlo

A composite of four components, each mapped onto [0, 1] before weighting.

Mapping to a common scale first is what makes the weights mean what they say.
Combining raw components — an RSI in [0, 100] with a breakout in [-0.2, 0.2] —
lets whichever has the widest natural range dominate regardless of its weight.

Standalone: no `portfolio_agent` import. Run `00_data_ingestion.ipynb` first so
the panel is cached.

## Setup

In [ ]:
# Dependencies. Torch is only needed by the two learned strategies (04, 05).
# !pip install -q pandas numpy pyarrow matplotlib huggingface_hub torch

import sys, pathlib

# afa_lab.py sits next to this notebook. On Colab (or anywhere the file is
# missing) fetch it from the repo — that is the only network call that touches
# GitHub, and nothing else here imports the portfolio_agent package.
if not pathlib.Path("afa_lab.py").exists():
    import urllib.request
    URL = ("https://raw.githubusercontent.com/3dwag98/afa/main/"
           "notebooks/standalone/afa_lab.py")
    urllib.request.urlretrieve(URL, "afa_lab.py")
    print("fetched afa_lab.py")

sys.path.insert(0, ".")
import afa_lab as L

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("toolkit loaded | torch available:", L.TORCH_AVAILABLE)

In [ ]:
# NSE large caps. Any symbol absent from the dataset is skipped rather than
# failing the run, so this list does not have to be exactly right.
UNIVERSE = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
    "HINDUNILVR",
    "ITC",
    "SBIN",
    "BHARTIARTL",
    "KOTAKBANK",
    "LT",
    "AXISBANK",
    "ASIANPAINT",
    "MARUTI",
    "SUNPHARMA",
    "TITAN",
    "ULTRACEMCO",
    "WIPRO",
    "NESTLEIND",
    "BAJFINANCE",
    "TATAMOTORS",
    "TATASTEEL",
    "POWERGRID",
    "NTPC",
    "ONGC",
    "HCLTECH",
    "JSWSTEEL",
    "GRASIM",
    "CIPLA",
    "COALINDIA"
]

START_DATE = "2018-01-01"
END_DATE   = None          # None = up to the dataset's last session
CACHE      = "data_cache"  # downloaded parquet files land here and are reused

print(len(UNIVERSE), "symbols requested")

In [ ]:
# Ingestion. One small parquet per symbol is pulled from the Hub dataset
# `vishnun0027/indian-market-historical-ohlcv` (2,421 NSE/BSE equities) and
# cleaned. Downloading per symbol rather than snapshotting the repo means a
# 30-name universe fetches 30 small files instead of 283 MB.
#
# Cleaning, in order: back-adjust OHLC by adj_close/close so a split is not read
# as a 90% crash, coerce numerics, drop unparseable dates and missing closes
# (rather than forward-filling, so a gap stays visible), and drop duplicate
# sessions keeping the last.
#
# If the Hub is unreachable the toolkit falls back to a synthetic panel and says
# so loudly. Synthetic results describe the generator, not the market.

panel = L.load_panel(UNIVERSE, start_date=START_DATE, end_date=END_DATE,
                     cache_dir=CACHE)

close = L.align_close_matrix(panel)
print(f"{len(panel)} symbols | {close.index.min().date()} -> {close.index.max().date()}"
      f" | {len(close)} sessions")

In [ ]:
# Features are computed per symbol and left NaN until each window has filled.
# They are never back-filled: a back-filled indicator is a look-ahead, and it is
# invisible in every metric downstream.
feature_panel = L.build_feature_panel(panel)

sample = feature_panel[sorted(feature_panel)[0]]
print(f"{len(sample.columns)} features:", list(sample.columns))
display(sample.dropna().tail(3))

In [ ]:
# Simulation settings, shared by every notebook so the strategies are comparable.
#
# execution_lag=1 is the property that keeps this honest: a signal computed from
# day t's close is traded into day t+1's return. The engine refuses lag=0.
config = L.BacktestConfig(
    initial_capital=1_000_000.0,
    cost_bps=25.0,        # all-in round trip for Indian cash equities
    max_weight=0.10,
    rebalance_days=5,     # weekly; the main control over turnover
    max_gross=1.0,        # long-only, unlevered
    execution_lag=1,
)

benchmark = L.equal_weight_benchmark(close, config)
print("equal-weight buy & hold:",
      {k: round(v, 4) for k, v in benchmark.stats.items()
       if k in ("cagr", "sharpe", "max_drawdown")})

## The four components

| Component | Reads | Mapped from |
| --- | --- | --- |
| Trend | `close / sma_200 - 1` | ±20%, saturating |
| Breakout | `close / high_20 - 1` | -10% to +5% |
| Volume | `volume / avg_volume_20` | 0.5x to 2.5x |
| Monte Carlo | P(higher in 21 sessions) | 0.30 to 0.70 |

The trailing high excludes today, so the breakout test is not self-referential —
comparing today's close against a window that contains it would make every new
high a breakout by construction.

The Monte Carlo term is a **block bootstrap** over trailing log returns, not a
Gaussian. Indian equity returns are fat-tailed and serially dependent enough
that a normal approximation understates both tails, and the left one is the
expensive side.

In [ ]:
params = L.RuleBasedParams(
    trend_weight=0.30,
    breakout_weight=0.25,
    volume_weight=0.20,
    mc_weight=0.25,
    min_score=0.55,            # composite needed to hold a name
    min_win_probability=0.50,
    require_trend=True,        # close > sma_200 is a hard gate, not a score
)

# The bootstrap is the slowest thing here. Set use_monte_carlo=False for a fast
# pass while you are changing weights; the gate then admits more names.
rule_based_scores = L.rule_based_scores(feature_panel, panel, params,
                                        use_monte_carlo=True)

held = (rule_based_scores > 0).sum(axis=1)
print(f"names passing the gate: mean {held.mean():.1f}, max {held.max()}, "
      f"zero on {(held == 0).mean():.1%} of sessions")

In [ ]:
# Where the score comes from, for one name. Each line is already on [0, 1], so
# the composite is directly readable as a weighted average.
symbol = sorted(feature_panel)[0]
f = feature_panel[symbol]

parts = pd.DataFrame({
    "trend": ((f["close"] / f["sma_200"] - 1).clip(-0.2, 0.2) + 0.2) / 0.4,
    "breakout": (f["breakout_20"].clip(-0.1, 0.05) + 0.1) / 0.15,
    "volume": (f["volume_ratio_20"].clip(0.5, 2.5) - 0.5) / 2.0,
}).dropna()

ax = parts.tail(250).plot(figsize=(11, 3.6), linewidth=1.1,
                          title=f"{symbol}: score components")
ax.axhline(params.min_score, color=L.PALETTE[1], linestyle="--", linewidth=1,
           label="min_score")
ax.grid(**L.GRID); ax.legend(ncol=4, fontsize=8)
import matplotlib.pyplot as plt; plt.tight_layout(); plt.show()

## Backtest

Against equal-weight buy-and-hold of the same names — the honest comparison for a long-only stock picker. Beating cash is not the question.

In [ ]:
result = L.run_backtest(rule_based_scores, close, config)

comparison = L.compare_stats({"rule based": result, "equal weight": benchmark})
display(comparison)

## Analysis

In [ ]:
L.plot_equity({"rule based": result}, title="rule based vs equal weight",
              benchmark=benchmark.returns)

In [ ]:
L.plot_return_profile(result, "rule based")

In [ ]:
L.plot_exposure(result, "rule based")

In [ ]:
L.plot_weight_heatmap(result, title="rule based: allocation over time")

## Sensitivity

The gate threshold is the parameter that matters most: it decides how often the
strategy is in the market at all. Sweeping it shows whether the result rests on
the signal or on one lucky cut point.

In [ ]:
sweep = {}
for threshold in (0.45, 0.50, 0.55, 0.60, 0.65):
    variant = L.RuleBasedParams(min_score=threshold)
    scores = L.rule_based_scores(feature_panel, panel, variant, use_monte_carlo=False)
    sweep[f"min_score={threshold}"] = L.run_backtest(scores, close, config)

table = L.compare_stats(sweep)
display(table[["sharpe", "cagr", "max_drawdown", "avg_positions", "ann_turnover"]])
L.plot_stats_table(table, title="Gate threshold sensitivity")

---

## What this does and does not show

Read before quoting any number above.

- **Survivorship.** The universe is today's large caps, applied to history. Names
  that were large caps in 2018 and are not now are absent, and they are absent
  precisely because they did badly. Every long-only result here is biased upward
  by an amount this notebook cannot measure. A point-in-time constituent list is
  the only fix, and this dataset does not carry one.
- **One universe, one period.** Thirty names over a few years is a single draw.
  The difference between two strategies here is well within what the draw alone
  could produce.
- **Costs are a flat 25 bps.** Real cost scales with size and with how illiquid
  the name is, and the fill is assumed at the close. A strategy whose edge is
  this side of costs is not distinguishable from one that has no edge.
- **No point-in-time fundamentals, no corporate actions beyond the price
  adjustment**, and no circuit-limit modelling. On Indian equities a
  circuit-locked session is untradeable, and the simulation will happily trade it.
- **Parameters were chosen, not fitted.** Nothing here is tuned on a held-out
  period. That is deliberate — tuning on this sample and reporting the result
  would be reporting the tuning.

The purpose of these notebooks is to make the mechanism legible and modifiable,
not to establish that any of these strategies makes money.